In [ ]:
import os
import pandas as pd
import sqlalchemy
import numpy as np
from sqlalchemy import text
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 DB 접속 정보 로드

engine = sqlalchemy.create_engine(
    "mysql+pymysql://{user}:{password}@{host}:{port}/{dbname}?charset=utf8".format(
        user     = os.getenv("DB_USER"),
        password = os.getenv("DB_PASSWORD"),
        host     = os.getenv("DB_HOST"),
        port     = os.getenv("DB_PORT"),
        dbname   = os.getenv("DB_NAME"),
    )
)

## 데이터 로드

In [ ]:
# 최근 1년 주문 데이터에서 유저별 상품 주문 빈도 집계
# 활성 회원(join_status=Y) + 판매 중인 상품(is_visible=Y) + 정상 주문만 포함
query = """
SELECT m.id AS user_id, oi.product_code, oi.product_name,
       o.order_date, oi.amount
FROM order_items oi
INNER JOIN orders o      ON oi.order_id = o.order_id
INNER JOIN members m     ON o.user_id = m.id
INNER JOIN products p    ON oi.product_code = p.product_code
WHERE o.status != '' AND o.status != 'OR'
  AND m.member_type != 'staff' AND m.join_status = 'Y'
  AND p.is_visible = 'Y'
  AND o.order_date >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR)
"""
df = pd.read_sql(query, engine)

In [3]:
order_cnt = (
    df.groupby(['id', 'product_code'])
      .size()
      .reset_index(name='frequency')
)

In [4]:
order_cnt['rank'] = (
    order_cnt
    .groupby('id')['frequency']
    .rank(method='dense', ascending=False)
)

In [5]:
order_cnt['log_date'] = pd.to_datetime('today').date()

In [ ]:
order_cnt.head(10)

In [ ]:
order_cnt.sort_values(by = 'rank').head(10)

## 결과 적재

> 유저별 상품 주문 빈도 순위를 `member_product_rank` 테이블에 저장합니다.  
> 매주 1회 스케줄 실행으로 자동 갱신됩니다.

In [ ]:
# 기존 데이터 초기화 후 새 순위 데이터 적재 (주 1회 스케줄 실행)
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE member_product_rank"))

order_cnt.to_sql(
    "member_product_rank",
    con=engine,
    if_exists="append",
    index=False
)

In [ ]:
# 적재 결과 확인
query = "SELECT * FROM member_product_rank ORDER BY user_id, rank"
rank_df = pd.read_sql(query, engine)
rank_df.head(20)

In [ ]:
# top만 뽑은거
top_items = order_cnt[order_cnt['rank'] <= 5] \
    .sort_values(['id', 'rank'])

top_items.head(20)